# L6 MODIS–VIIRS harmonization validation

Run after notebook 02. HAR-01/02 read harmonized processed files one year at a time. HAR-03 through HAR-07 read only `metrics/l1_cal_monthly.parquet`. Calibration defaults to 2018–2021 and test evaluation starts in 2022.

The factor maps MODIS to VIIRS using the ratio of calibration sums. Its leave-one-year-out range and jackknife standard error describe sensitivity to calibration years. They are not a formal confidence interval.

Use the `metricas-pipeline` branch. For private repository access, enable a `GITHUB_TOKEN` Colab secret. Configure the Drive path and year selection below. A selected year requires both processed sensor files. An empty sensor file is supported.


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import base64
import os
import subprocess
import sys

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Sprinklers')
REPO = Path('/content/repo')
environment = os.environ.copy()
try:
    token = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    token = None
if token:
    credential = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    environment.update(GIT_CONFIG_COUNT='1',
                       GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
                       GIT_CONFIG_VALUE_0=f'AUTHORIZATION: basic {credential}')
    del credential
if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], env=environment, check=True)
    subprocess.run(['git', '-C', str(REPO), 'switch', 'metricas-pipeline'], env=environment, check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'metricas-pipeline'],
                   env=environment, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'metricas-pipeline',
                    'https://github.com/Rui-Martins31/NASA_Space_Apps_2026_Sprinklers.git',
                    str(REPO)], env=environment, check=True)
del token, environment
sys.path.insert(0, str(REPO / 'data' / 'pipeline'))


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib
import json
import re
import subprocess
import pandas as pd
from metrics import harmonization_overlap, harmonization_factor
from metrics.calendar_metrics import eligible_cell_days
from metrics.common import GRID_DEG as DEFAULT_GRID_DEG

importlib.reload(harmonization_overlap)
importlib.reload(harmonization_factor)
CAL_START = 2018
CAL_END = 2021
TEST_START = 2022
MIN_PAIRS = 12
MIN_CAL_MONTHS = 24
MIN_CAL_YEARS = 3
MAX_UNKNOWN_DAYS = 0
GRID_DEG = DEFAULT_GRID_DEG
YEARS = None  # None selects available processed years >= CAL_START.
FORCE = False
PROCESSED = ROOT / 'processed'
METRICS = ROOT / 'metrics'
PARTIAL = METRICS / '_partial_lote6'
COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
PERIOD_OPTIONS = dict(cal_start=CAL_START, cal_end=CAL_END,
                      test_start=TEST_START, max_unknown_days=MAX_UNKNOWN_DAYS)
PARAMETERS = dict(PERIOD_OPTIONS, min_pairs=MIN_PAIRS, min_cal_months=MIN_CAL_MONTHS,
                  min_cal_years=MIN_CAL_YEARS, grid_deg=GRID_DEG, years=YEARS)
PROCESSED_COLUMNS = ['country', 'instrument', 'satellite', 'latitude', 'longitude',
                     'acq_date', 'type', 'confidence_class']
MONTHLY_COLUMNS = ['country', 'year', 'month', 'series', 'cell_days', 'partial', 'n_unknown_days']
OVERLAP_COLUMNS = ['country', 'year', 'month', 'n_modis', 'n_viirs', 'n_both', 'n_union']

def atomic_parquet(frame, path):
    temporary = path.with_suffix('.parquet.tmp')
    frame.to_parquet(temporary, index=False)
    temporary.replace(path)

def atomic_json(value, path):
    temporary = path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(value, sort_keys=True), encoding='utf-8')
    temporary.replace(path)


In [ ]:
monthly_path = METRICS / 'l1_cal_monthly.parquet'
if not monthly_path.is_file():
    raise FileNotFoundError(f'Missing {monthly_path}. Run 02_metrics_lote1.ipynb first.')
if YEARS is None:
    years = set()
    for path in PROCESSED.glob('*.parquet'):
        match = re.fullmatch(r'(modis|viirs-jpss1)_([0-9]{4})', path.stem)
        if match and int(match[2]) >= CAL_START:
            years.add(int(match[2]))
    years = sorted(years)
else:
    years = sorted({int(year) for year in YEARS if int(year) >= CAL_START})
if not years:
    raise FileNotFoundError(f'No selected processed sensor-year files in {PROCESSED}')
sources = {year: [PROCESSED / f'{sensor}_{year}.parquet'
                  for sensor in ['modis', 'viirs-jpss1']] for year in years}
missing = [str(path) for paths in sources.values() for path in paths if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing processed counterparts: ' + ', '.join(missing))
PARTIAL.mkdir(parents=True, exist_ok=True)
monthly = pd.read_parquet(monthly_path, columns=MONTHLY_COLUMNS)
# Validate monthly keys and counts before processing the larger inputs.
harmonization_factor.comparable_pairs(monthly, **PERIOD_OPTIONS)
module_digest = hashlib.sha256(Path(harmonization_overlap.__file__).read_bytes()).hexdigest()


In [ ]:
parts = []
for year in years:
    partial_path = PARTIAL / f'har01_{year}.parquet'
    marker = PARTIAL / f'har01_{year}.json'
    signature = dict(cal_start=CAL_START, grid_deg=GRID_DEG, code=module_digest,
                     commit=COMMIT, sources=[dict(name=path.name, size=path.stat().st_size,
                                                  mtime_ns=path.stat().st_mtime_ns)
                                           for path in sources[year]])
    cached = None
    if not FORCE and partial_path.is_file() and marker.is_file():
        try:
            if json.loads(marker.read_text(encoding='utf-8')) == signature:
                cached = pd.read_parquet(partial_path, columns=OVERLAP_COLUMNS)
        except (OSError, ValueError):
            pass
    if cached is not None:
        parts.append(cached)
        print(f'Reusing completed overlap for {year}')
        continue
    marker.unlink(missing_ok=True)
    compact = []
    for instrument, path in zip(['MODIS', 'VIIRS'], sources[year]):
        frame = pd.read_parquet(path, columns=PROCESSED_COLUMNS)
        if not frame.instrument.astype('string').str.upper().eq(instrument).fillna(False).all():
            raise ValueError(f'Unexpected instrument in {path}')
        for column in ['country', 'instrument', 'satellite', 'confidence_class']:
            frame[column] = frame[column].astype('category')
        cell_days = eligible_cell_days(frame, grid_deg=GRID_DEG)
        if not cell_days.date.dt.year.eq(year).all():
            raise ValueError(f'Acquisition dates outside filename year in {path}')
        compact.append(cell_days)
        del frame
    cell_days = pd.concat(compact, ignore_index=True)
    overlap = harmonization_overlap.har01_cellday_overlap(cell_days, cal_start=CAL_START)
    atomic_parquet(overlap, partial_path)
    atomic_json(signature, marker)
    parts.append(overlap)
    del compact, cell_days
    print(f'Processed overlap for {year}')
overlap = pd.concat(parts, ignore_index=True).sort_values(['country', 'year', 'month']).reset_index(drop=True)


In [ ]:
tables = {
    'l6_har01_overlap': overlap,
    'l6_har02_jaccard': harmonization_overlap.har02_jaccard(overlap),
    'l6_har03_comparable_months': harmonization_factor.har03_comparable_months(monthly, **PERIOD_OPTIONS),
    'l6_har04_pearson': harmonization_factor.har04_pearson(monthly, min_pairs=MIN_PAIRS, **PERIOD_OPTIONS),
    'l6_har05_spearman': harmonization_factor.har05_spearman(monthly, min_pairs=MIN_PAIRS, **PERIOD_OPTIONS),
    'l6_har06_factor': harmonization_factor.har06_factor(monthly, min_cal_months=MIN_CAL_MONTHS, **PERIOD_OPTIONS),
    'l6_har07_factor_uncertainty': harmonization_factor.har07_factor_uncertainty(
        monthly, min_cal_months=MIN_CAL_MONTHS, min_cal_years=MIN_CAL_YEARS, **PERIOD_OPTIONS),
}
log = []
executed_at = datetime.now(timezone.utc).isoformat()
for name, result in tables.items():
    atomic_parquet(result, METRICS / f'{name}.parquet')
    log.append(dict(metric=name, parameters=json.dumps(PARAMETERS, sort_keys=True),
                    overlap_years=json.dumps(years), commit=COMMIT, output_rows=len(result),
                    executed_at_utc=executed_at))
log_path = METRICS / '_log_lote6.csv'
temporary = log_path.with_suffix('.csv.tmp')
pd.DataFrame(log).to_csv(temporary, index=False)
temporary.replace(log_path)


In [ ]:
assert overlap.n_union.equals(overlap.n_modis + overlap.n_viirs - overlap.n_both)
jaccard = tables['l6_har02_jaccard'].jaccard.dropna()
assert jaccard.between(0, 1).all()
for name, result in tables.items():
    print(name, result.shape)
    print(result.dtypes)
    print(result.head(3).to_string(index=False))
print(f'L6 completed: {len(tables)} tables written to {METRICS}')
